In [1]:
import pandas as pd
import numpy as np
import re
import os

OUT_DIR = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}

stability_df_cpd = pd.read_csv(os.path.join(OUT_DIR, "cpd_stability_results.csv"))

shortlist_cpd_80 = stability_df_cpd[stability_df_cpd["stability_fraction"] >= 0.80].copy()
shortlist_cpd_80["core_name"] = shortlist_cpd_80["probe_id"].map(strip_suffix)

pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_cpd_80 = shortlist_cpd_80.merge(pos_lookup, on="core_name", how="left")
shortlist_cpd_80 = shortlist_cpd_80[~shortlist_cpd_80["Chr"].isin(non_autosomal)].copy()

print("Shortlist at >=80% stability:", len(shortlist_cpd_80))

Shortlist at >=80% stability: 154


In [2]:
import pandas as pd
import numpy as np
import re
import os
import gc

AA_GENO = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"

# rebuild smoker_cols_cpd (same filter as original: smokers with valid CPD)
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])

encoded_df = pd.read_csv(AA_GENO)
all_sample_ids = encoded_df.columns[1:].tolist()
smoker_id_set = set(smokers["sample_id"].tolist())
smoker_cols_cpd = [s for s in all_sample_ids if s in smoker_id_set]

probe_rows = encoded_df[encoded_df["probe_id"].isin(set(shortlist_cpd_80["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist_cpd_80["probe_id"].tolist())
X_shortlist_cpd80 = probe_rows[smoker_cols_cpd].to_numpy(dtype=np.float64).T
probe_id_to_idx_cpd80 = {pid: i for i, pid in enumerate(shortlist_cpd_80["probe_id"].tolist())}
del encoded_df, probe_rows
gc.collect()

def get_geno_cpd80(pid):
    return X_shortlist_cpd80[:, probe_id_to_idx_cpd80[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000, get_geno=None):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_cpd80 = greedy_ld_prune(shortlist_cpd_80, get_geno=get_geno_cpd80)
shortlist_pruned_cpd80 = shortlist_cpd_80[
    shortlist_cpd_80["probe_id"].isin(retained_cpd80)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_cpd80)} SNPs")

After LD pruning: 139 SNPs


In [3]:
import pandas as pd
import numpy as np
import os

AA_GENO = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
OUT_DIR = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"

encoded_df2 = pd.read_csv(AA_GENO)
probe_rows2 = encoded_df2[encoded_df2["probe_id"].isin(set(shortlist_pruned_cpd80["probe_id"]))].copy()
probe_rows2 = probe_rows2.set_index("probe_id").reindex(shortlist_pruned_cpd80["probe_id"].tolist())
X_check = probe_rows2[smoker_cols_cpd].to_numpy(dtype=np.float64).T

corr_matrix = np.corrcoef(X_check.T)

to_remove = set()
for i in range(len(shortlist_pruned_cpd80)):
    for j in range(i+1, len(shortlist_pruned_cpd80)):
        if abs(corr_matrix[i,j]) > 0.99 and shortlist_pruned_cpd80["probe_id"].iloc[j] not in to_remove:
            to_remove.add(shortlist_pruned_cpd80["probe_id"].iloc[j])

print(f"Perfectly correlated SNPs to remove: {len(to_remove)}")

shortlist_final_cpd80 = shortlist_pruned_cpd80[
    ~shortlist_pruned_cpd80["probe_id"].isin(to_remove)
].copy()
print(f"Final shortlist: {len(shortlist_final_cpd80)} SNPs")

shortlist_final_cpd80.to_csv(os.path.join(OUT_DIR, "cpd_shortlist_final_80pct.csv"), index=False)
print("Saved.")

Perfectly correlated SNPs to remove: 67
Final shortlist: 72 SNPs
Saved.


In [4]:
import numpy as np

# genotype counts within smoker-only subset for the 139 shortlisted SNPs
maf_smoker_subset = []
for pid in shortlist_pruned_cpd80["probe_id"]:
    g = get_geno_cpd80(pid)
    # minor allele frequency proxy: mean/2 (since 0/1/2 encoding)
    freq = g.mean() / 2
    maf = min(freq, 1 - freq)
    maf_smoker_subset.append(maf)

shortlist_pruned_cpd80["maf_in_smokers"] = maf_smoker_subset
print(shortlist_pruned_cpd80["maf_in_smokers"].describe())
print("\nSNPs with MAF < 0.05 in smoker subset:", (shortlist_pruned_cpd80["maf_in_smokers"] < 0.05).sum())
print("SNPs with MAF < 0.02 in smoker subset:", (shortlist_pruned_cpd80["maf_in_smokers"] < 0.02).sum())

count    139.000000
mean       0.011409
std        0.047671
min        0.000307
25%        0.000307
50%        0.000613
75%        0.001533
max        0.386266
Name: maf_in_smokers, dtype: float64

SNPs with MAF < 0.05 in smoker subset: 131
SNPs with MAF < 0.02 in smoker subset: 131


In [6]:
import numpy as np
import json
import os
import pandas as pd

OUT_DIR = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"
AA_META = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"

# reload Y_cpd aligned to smoker_cols_cpd
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers_aligned = meta_df.set_index("sample_id").reindex(smoker_cols_cpd).reset_index()
smokers_aligned["cpd"] = pd.to_numeric(smokers_aligned["cpd"], errors="coerce")
Y_cpd80 = smokers_aligned["cpd"].values.astype(np.float64)

pruned_ids_final80 = shortlist_final_cpd80["probe_id"].tolist()
col_names_cpd80 = pruned_ids_final80 + ["CPD"]

# pull columns directly from the already-loaded X_shortlist_cpd80 (139 SNPs)
# using probe_id_to_idx_cpd80 to find each final SNP's column index
final_col_indices = [probe_id_to_idx_cpd80[pid] for pid in pruned_ids_final80]
X_pc_cpd80 = X_shortlist_cpd80[:, final_col_indices]

X_pc_full_cpd80 = np.hstack([X_pc_cpd80, Y_cpd80.reshape(-1, 1)])

print("PC input shape:", X_pc_full_cpd80.shape)

np.save(os.path.join(OUT_DIR, "cpd_pc_input_80pct.npy"), X_pc_full_cpd80)
with open(os.path.join(OUT_DIR, "cpd_pc_col_names_80pct.json"), "w") as f:
    json.dump(col_names_cpd80, f)
print("Saved.")

PC input shape: (1631, 73)
Saved.
